In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

In [ ]:
ROOT = Path.cwd()
TASKS = ['a_import', 'b_diagnose', 'c_plot']
for task in TASKS:
    (ROOT/task/'output').mkdir(parents=True, exist_ok=True)
(ROOT/'a_import/input').mkdir(parents=True, exist_ok=True)

for previous, task in zip(TASKS, TASKS[1:]):
    link = ROOT/task/'input'
    if not link.is_symlink():
        link.symlink_to(Path('..')/previous/'output', target_is_directory=True)

plan = [path for task in TASKS for path in sorted((ROOT/task/'src').glob('*.ipynb'))]
display(pd.DataFrame({'notebook': [str(path.relative_to(ROOT)) for path in plan]}))

In [ ]:
for path in plan:
    scope = {'__name__': '__main__'}
    os.chdir(path.parent)
    try:
        for cell in json.loads(path.read_text())['cells']:
            exec(compile(''.join(cell['source']), str(path.relative_to(ROOT)), 'exec'), scope)
    finally:
        os.chdir(ROOT)